# T5Gemma with CTranslate2

This notebook demonstrates how to use T5Gemma models with CTranslate2 for efficient inference.

T5Gemma combines the encoder-decoder architecture based on Gemma 2:
- Pre+post layer normalization (Gemma 2 style)
- Grouped Query Attention (GQA)
- GeGLU activation
- RMSNorm
- Encoder and decoder both use Gemma 2 decoder architecture (encoder with bidirectional attention)

**Note**: T5Gemma support requires CTranslate2 with encoder-decoder pre+post layer norm support. You'll need to install the custom wheel from this PR.

## Installation

Install the custom CTranslate2 wheel with T5Gemma support:

In [ ]:
# Install transformers with T5Gemma support (requires recent version)
!pip install -q 'transformers>=4.50.0' torch sentencepiece

# Download and install the custom CTranslate2 wheel with T5Gemma support
# This wheel includes encoder-decoder pre+post layer norm support
import urllib.request

wheel_url = "https://github.com/jncraton/CTranslate2/releases/download/v4.6.2-t5gemma/ctranslate2-4.6.2-cp310-cp310-linux_x86_64.whl"
wheel_file = "/tmp/ctranslate2-4.6.2-cp310-cp310-linux_x86_64.whl"

print("Downloading CTranslate2 wheel with T5Gemma support...")
try:
    urllib.request.urlretrieve(wheel_url, wheel_file)
    print(f"✓ Downloaded to {wheel_file}")
    !pip install {wheel_file}
    print("✓ CTranslate2 with T5Gemma support installed!")
except Exception as e:
    print(f"✗ Error downloading wheel: {e}")
    print("\nAlternative: Build from source or wait for official release")
    print("See: https://github.com/jncraton/CTranslate2/pull/XXX")

## Verify Installation

In [ ]:
import ctranslate2
from ctranslate2.converters.transformers import _MODEL_LOADERS

# Check if T5Gemma support is available
if "T5GemmaConfig" in _MODEL_LOADERS:
    print("✓ T5Gemma support is available!")
    print(f"  Loader: {_MODEL_LOADERS['T5GemmaConfig'].__name__}")
else:
    print("✗ T5Gemma support is not available")
    print("  Make sure you installed the correct CTranslate2 wheel")

## Convert T5Gemma Model

We'll use the `harshaljanjani/tiny-t5gemma-test` model for demonstration (it's small and quick to download):

In [ ]:
model_name = "harshaljanjani/tiny-t5gemma-test"
output_dir = "ct2_t5gemma_model"

print(f"Converting {model_name}...")

converter = ctranslate2.converters.TransformersConverter(
    model_name,
    trust_remote_code=True
)

converted_model_path = converter.convert(output_dir, quantization="int8")
print(f"✓ Model converted successfully to: {converted_model_path}")

## Load Tokenizer

Load the tokenizer from the original model:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("✓ Tokenizer loaded")

## Run Inference

Now let's test the converted model with some example inputs:

In [ ]:
# Load the converted model
translator = ctranslate2.Translator(converted_model_path)
print(f"✓ Model loaded (device: {translator.device}, compute_type: {translator.compute_type})")

# Example 1: Translation
source_text = "translate English to German: The house is wonderful."
source_tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(source_text))

results = translator.translate_batch([source_tokens])

# Decode the result
target_tokens = results[0].hypotheses[0]
target_text = tokenizer.decode(
    tokenizer.convert_tokens_to_ids(target_tokens),
    skip_special_tokens=True
)

print("\nTranslation Result:")
print("=" * 60)
print(f"Source: {source_text}")
print(f"Target: {target_text}")
print("=" * 60)

## Batch Inference

CTranslate2 efficiently handles batch processing:

In [ ]:
# Multiple translation examples
source_texts = [
    "translate English to French: Hello world",
    "translate English to German: Good morning",
    "translate English to Spanish: Thank you",
]

# Tokenize all inputs
source_tokens_batch = [
    tokenizer.convert_ids_to_tokens(tokenizer.encode(text))
    for text in source_texts
]

# Run batch translation
results = translator.translate_batch(source_tokens_batch)

# Display results
print("\nBatch Translation Results:")
print("=" * 60)
for source, result in zip(source_texts, results):
    target_tokens = result.hypotheses[0]
    target_text = tokenizer.decode(
        tokenizer.convert_tokens_to_ids(target_tokens),
        skip_special_tokens=True
    )
    print(f"Source: {source}")
    print(f"Target: {target_text}")
    print("-" * 60)

## Advanced: Beam Search

CTranslate2 supports various decoding strategies:

In [ ]:
source_text = "translate English to German: The house is wonderful."
source_tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(source_text))

# Beam search with multiple hypotheses
results = translator.translate_batch(
    [source_tokens],
    beam_size=5,
    num_hypotheses=3,
    return_scores=True
)

print("\nBeam Search Results:")
print("=" * 60)
for i, (hypothesis, score) in enumerate(zip(results[0].hypotheses, results[0].scores)):
    target_text = tokenizer.decode(
        tokenizer.convert_tokens_to_ids(hypothesis),
        skip_special_tokens=True
    )
    print(f"Hypothesis {i+1} (score: {score:.4f}): {target_text}")